# Brawl Stars — live data → EDA → real model training → report
Minimal code-first CS156 pipeline. Binary WIN/LOSS because no DRAW observations exist yet.

In [ ]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, log_loss,
                             brier_score_loss, ConfusionMatrixDisplay)
from sklearn.base import clone
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SHEET_ID = "1PT7GDSb56HKSjGbvmj5-bGVWCmbP8Wo7-tSkRxEk7aQ"
BLUE = ["Blue_1", "Blue_2", "Blue_3"]
RED = ["Red_1", "Red_2", "Red_3"]
SLOTS = BLUE + RED
SEED = 42
pd.set_option("display.max_columns", 30)
print("Assignment root:", ROOT.resolve())

## 1. Read the Google Sheet
For **live Google Sheets**, install `gspread google-auth-oauthlib` and set up a *Google Cloud OAuth Desktop App* client in `credentials.json` in this assignment folder. Run `gspread.oauth()` once; your browser will ask you to authorize. `DATA_SOURCE='auto'` uses live Google data when that credentials file exists; otherwise it transparently uses the committed CSV snapshot. Set `'google'` to require live data (and fail if not configured). Never commit the credentials or OAuth token.

In [ ]:
DATA_SOURCE = "auto"  # "google" requires OAuth, "snapshot" uses frozen CSV
OAUTH_CLIENT = ROOT / "credentials.json"
OAUTH_TOKEN = ROOT / ".google_token.json"
SNAPSHOT = ROOT / "data" / "raw_matches.csv"

def read_google_sheet():
    import gspread
    client = gspread.oauth(credentials_filename=str(OAUTH_CLIENT),
                           authorized_user_filename=str(OAUTH_TOKEN),
                           scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"])
    worksheet = client.open_by_key(SHEET_ID).worksheet("Battles")
    return pd.DataFrame(worksheet.get_all_records())

if DATA_SOURCE == "google" or (DATA_SOURCE == "auto" and OAUTH_CLIENT.exists()):
    if not OAUTH_CLIENT.exists():
        raise FileNotFoundError(f"Google OAuth client missing: {OAUTH_CLIENT}")
    raw = read_google_sheet()
    source_used = "LIVE GOOGLE SHEETS"
else:
    raw = pd.read_csv(SNAPSHOT)
    source_used = "FROZEN CSV SNAPSHOT (not live)"
print("DATA SOURCE:", source_used)
print("Rows:", len(raw))
display(raw.head(4))

## 2. Check & clean before modeling

In [ ]:
df = raw.copy()
required = ["Match_ID", *SLOTS, "Blue_Win", "Result"]
assert set(required).issubset(df), f"Missing columns: {set(required)-set(df)}"
df = df.replace(r"^\\s*$", np.nan, regex=True)
assert not df[required].isna().any().any(), "Missing essential cells"
for col in SLOTS:
    df[col] = df[col].astype(str).str.strip().str.upper().str.replace(r"\\s+", "_", regex=True)
    df[col] = df[col].replace({"COLLETTE": "COLETTE"})
df["Blue_Win"] = pd.to_numeric(df["Blue_Win"], errors="raise").astype(int)
assert df["Blue_Win"].isin([0, 1]).all(), "Unexpected result label (draws need separate workflow)"
assert (df["Result"].str.upper().map({"VICTORY":1,"DEFEAT":0}) == df["Blue_Win"]).all()
assert df["Match_ID"].is_unique, "Repeated match IDs"
assert all(df[side].nunique(axis=1).eq(3).all() for side in (BLUE,RED))
vocab = sorted(pd.unique(df[SLOTS].to_numpy().ravel()))
print(f"Validated {len(df)} matches; {len(vocab)} unique characters")
print("Outcomes:",df["Blue_Win"].value_counts().sort_index().to_dict())
if "Map" in df: print("Maps:", df["Map"].value_counts().to_dict())
if "Mode" in df: print("Modes:", df["Mode"].value_counts().to_dict())

## 3. Quick data analysis

In [ ]:
freq = pd.Series(df[SLOTS].to_numpy().ravel()).value_counts()
fig, axes = plt.subplots(1,2,figsize=(13,4))
freq.head(14).plot.bar(ax=axes[0], title="Top 14 brawler appearances")
axes[0].set_ylabel("Appearances across both teams")
df["Blue_Win"].map({0:"Loss",1:"Win"}).value_counts().reindex(["Win","Loss"]).plot.bar(ax=axes[1],title="Observed outcome balance")
axes[1].set_ylabel("Matches")
plt.tight_layout()
plt.show()
display(freq.rename("appearances").to_frame().head(20))

## 4. Features (composition, not portrait order)
Signed representation: +1 for a Blue brawler, −1 for a Red brawler. Separate-side representation: two indicator vectors. `Blue_1` is only a **proxy**, not necessarily your personal brawler.

In [ ]:
def encode_comps(frame, labels, style="signed"):
    idx = {b:i for i,b in enumerate(labels)}
    size = len(labels) if style=="signed" else 2*len(labels)
    x = np.zeros((len(frame),size), dtype=float)
    for row_idx,(_,row) in enumerate(frame.iterrows()):
        for c in BLUE: x[row_idx,idx[row[c]]] += 1
        for c in RED: x[row_idx,idx[row[c]] + (len(labels) if style=="separate" else 0)] += (-1 if style=="signed" else 1)
    return x

X_signed = encode_comps(df,vocab,"signed")
X_separate = encode_comps(df,vocab,"separate")
X_single = pd.get_dummies(df["Blue_1"]).reindex(columns=vocab, fill_value=0).to_numpy(dtype=float)
y = df["Blue_Win"].to_numpy()
assert X_signed.shape == (len(df),len(vocab))
assert X_separate.shape == (len(df),2*len(vocab))
reorder=df.copy()
reorder[BLUE] = df[BLUE[::-1]].to_numpy()
assert np.array_equal(encode_comps(reorder,vocab),X_signed)
print("Feature shapes:", X_signed.shape,X_separate.shape,X_single.shape)

## 5. ACTUAL training: five-fold out-of-fold model comparisons
Every fold **fits new model parameters** using 80% of the matches and predicts the other 20%. No model is fitted on those five validation labels. This is small-sample exploratory validation, not a stable generalization estimate.

In [ ]:
models = {
    "Class prior": (np.zeros((len(df),1)), DummyClassifier(strategy="prior")),
    "Blue slot 1 only": (X_single, LogisticRegression(C=0.1,max_iter=3000)),
    "Full comp (signed)": (X_signed, LogisticRegression(C=0.1,max_iter=3000)),
    "Full comp (separate)": (X_separate, LogisticRegression(C=0.1,max_iter=3000)),
}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
oof = {}
fitted_folds = {}
for name,(X,estimator) in models.items():
    p=np.full(len(y),np.nan)
    fitted_folds[name]=[]
    for train_idx,valid_idx in cv.split(X,y):
        model=clone(estimator).fit(X[train_idx],y[train_idx])  # REAL FIT
        p[valid_idx]=model.predict_proba(X[valid_idx])[:,list(model.classes_).index(1)]
        fitted_folds[name].append(model)
    oof[name]=p
assert all(np.isfinite(p).all() for p in oof.values())
scores=pd.DataFrame([{
    "Model":name,
    "Log loss ↓":log_loss(y,np.column_stack([1-p,p]),labels=[0,1]),
    "Brier ↓":brier_score_loss(y,p),
    "Accuracy ↑":accuracy_score(y,p>=0.5),
    "Balanced accuracy ↑":balanced_accuracy_score(y,p>=0.5),
} for name,p in oof.items()]).sort_values("Log loss ↓")
display(scores.round(4))

## 6. Performance graphs — out-of-fold predictions only

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
scores.sort_values("Log loss ↓",ascending=False).plot.barh(x="Model",y="Log loss ↓",ax=axes[0],legend=False,title="Cross-validated log loss (lower is better)")
axes[0].set_xlabel("Log loss")
scores.sort_values("Brier ↓",ascending=False).plot.barh(x="Model",y="Brier ↓",ax=axes[1],legend=False,title="Cross-validated Brier score (lower is better)")
axes[1].set_xlabel("Brier score")
plt.tight_layout();plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,4))
main_p=oof["Full comp (signed)"]
ConfusionMatrixDisplay.from_predictions(y,main_p>=.5,display_labels=["Loss","Win"],ax=axes[0],colorbar=False)
axes[0].set_title("Out-of-fold predictions")
axes[1].scatter(np.arange(len(y)),main_p,c=y,cmap="coolwarm",vmin=0,vmax=1,s=60)
axes[1].axhline(.5,ls="--",color="gray")
axes[1].set(ylim=(0,1),xlabel="Match index",ylabel="Predicted P(Win)",title="Blue win probability (red=loss, blue=win)")
plt.tight_layout();plt.show()

## 7. Tiny step-size demonstration: SGD model fitting over epochs
This plot tracks **training loss**, not cross-validated performance. The early iterations should show actual optimization, but decreasing training loss alone does not imply useful win probabilities.

In [ ]:
X_train,X_val,y_train,y_val=train_test_split(X_signed,y,test_size=.24,random_state=SEED,stratify=y)
sgd=SGDClassifier(loss="log_loss",penalty="l2",alpha=.1,learning_rate="constant",eta0=.001,random_state=SEED)
hist_train,hist_val=[],[]
for epoch in range(1,501):
    sgd.partial_fit(X_train,y_train,classes=np.array([0,1]))
    tr=sgd.predict_proba(X_train)[:,1]
    va=sgd.predict_proba(X_val)[:,1]
    hist_train.append(log_loss(y_train,np.column_stack([1-tr,tr]),labels=[0,1]))
    hist_val.append(log_loss(y_val,np.column_stack([1-va,va]),labels=[0,1]))
plt.figure(figsize=(8,4))
plt.plot(hist_train,label="Training log loss")
plt.plot(hist_val,label="Validation log loss")
plt.xlabel("Epoch (learning rate = 0.001)")
plt.ylabel("Log loss")
plt.title("Gradient updates: true learning curve, small training sample")
plt.legend();plt.tight_layout();plt.show()
print("SGD loss:",round(hist_train[-1],4),"train;",round(hist_val[-1],4),"validation")

## 8. Fit the final model on **all** known matches and predict a proposed matchup
Do not evaluate this model on the same 25 training matches; evaluation above used held-out predictions. Unknown characters require more training data.

In [ ]:
final_model=LogisticRegression(C=0.1,max_iter=3000,random_state=SEED).fit(X_signed,y)
def predict_match(blue,red):
    if len(blue)!=3 or len(red)!=3 or len(set(blue))!=3 or len(set(red))!=3:
        raise ValueError("Each team requires 3 different brawlers")
    blue=[str(x).strip().upper().replace(" ","_") for x in blue]
    red=[str(x).strip().upper().replace(" ","_") for x in red]
    unknown=(set(blue)|set(red))-set(vocab)
    if unknown: raise ValueError(f"Unseen brawlers: {sorted(unknown)}")
    query=pd.DataFrame([{**dict(zip(BLUE,blue)),**dict(zip(RED,red))}])
    prob=float(final_model.predict_proba(encode_comps(query,vocab))[0,list(final_model.classes_).index(1)])
    return {"Win":prob,"Loss":1-prob,"Draw":None}
display(predict_match(df.iloc[0][BLUE].tolist(),df.iloc[0][RED].tolist()))

## 9. Machine-generated results report

In [ ]:
best=scores.iloc[0]
baseline=scores.loc[scores["Model"]=="Class prior"].iloc[0]
full=scores.loc[scores["Model"]=="Full comp (signed)"].iloc[0]
print("="*58)
print("BRAWL STARS — CURRENT PIPELINE RESULTS")
print("="*58)
print("Data:",source_used)
print("Games:",len(df),"| Win:",int(y.sum()),"| Loss:",int(len(y)-y.sum()),"| Draw: 0")
print("Characters:",len(vocab),"| Features: signed",X_signed.shape[1],"vs separate",X_separate.shape[1])
print("\nCross-validated comparisons (five-fold):")
print(scores.to_string(index=False,float_format=lambda n:f"{n:.4f}"))
print("\nLowest log loss:",best["Model"])
print(f"Signed composition vs baseline log-loss difference: {full['Log loss ↓']-baseline['Log loss ↓']:+.4f}")
print("\nWARNING: only 25 games; same map/mode; confounded by player skill. No draw class.")
report_path=ROOT/"model_results.csv"
scores.to_csv(report_path,index=False)
print("Saved:",report_path)
print("Fitted final binary model:",type(final_model).__name__,"on",len(y),"matches")